# Aula 3 — Avaliação de sistemas agênticos

Um agente real pode variar entre execuções. Por isso, a avaliação deve comparar propriedades,
não frases completas.

Avaliaremos contrato, evidências, segurança e expectativas de cenário.

In [ ]:
from pathlib import Path
import json, os, sys
from dotenv import load_dotenv

def find_course_root(start_path=None):
    current = Path(start_path or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "target_project").is_dir():
            return candidate
    raise FileNotFoundError("Nao foi encontrada a pasta target_project/.")

COURSE_ROOT = find_course_root()
TARGET_PROJECT = COURSE_ROOT / "target_project"
AULA_3_DIR = COURSE_ROOT / "Aula 3"

if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

dotenv_candidates = [AULA_3_DIR / ".env", COURSE_ROOT / ".env"]
for env_path in dotenv_candidates:
    if env_path.exists():
        load_dotenv(env_path)
        break

MODEL_NAME = os.getenv("MODEL_NAME", "openai/gpt-4o-mini")

## Preparação independente

Nesta etapa, configuramos o ambiente e os insumos que serão compartilhados por todos os cenários.

Lógica desta preparação:
1. resolver caminhos do curso e carregar variáveis de ambiente;
2. montar o catálogo do projeto para obter IDs válidos de evidência;
3. inicializar o modelo com baixa variabilidade (temperature=0);
4. carregar os casos de avaliação a partir de um arquivo determinístico.

Objetivo pedagógico: separar preparação de execução para deixar explícito o que é estrutura fixa
e o que muda entre cenários de teste.

In [11]:
from shared.aula_3.project_access import build_project_catalog
from shared.aula_3.crewai_components import (
    build_investigation_crew, build_llm, build_project_tools, parse_crew_json
)
from shared.aula_3.validation import evaluate_analysis

catalog = build_project_catalog(TARGET_PROJECT)
valid_evidence_ids = {d["evidence_id"] for d in catalog["documents"]}
llm = build_llm(MODEL_NAME, temperature=0)

cases = json.loads(
    (AULA_3_DIR / "data/evaluation_cases/cases.json").read_text(encoding="utf-8")
)
cases

[{'case_id': 'case_timeout',
  'incident': {'incident_id': 'EVAL-001',
   'title': 'Pipeline excedeu o tempo limite',
   'description': 'A execução terminou por timeout após aumento de volume.'},
  'expectations': {'requires_evidence': True, 'minimum_confidence': 0.5}},
 {'case_id': 'case_insufficient_information',
  'incident': {'incident_id': 'EVAL-002',
   'title': 'Falha não identificada',
   'description': 'O serviço falhou sem mensagem, log associado ou alteração conhecida.'},
  'expectations': {'requires_human_review': True, 'maximum_confidence': 0.6}}]

## Expectativas por propriedades

A avaliação de cenário não compara frases literais do modelo. Em vez disso, ela verifica
propriedades observáveis da saída.

O bloco de código abaixo transforma expectativas em checks booleanos, como:
- presença obrigatória de evidências;
- exigência de revisão humana;
- faixa mínima e máxima de confiança.

Essa abordagem aumenta a robustez da avaliação para respostas semanticamente equivalentes,
mesmo quando o texto final varia entre execuções.

In [12]:
def safe_float(value, default):
    try:
        return float(value)
    except (TypeError, ValueError):
        return float(default)


def evaluate_case_expectations(payload, expectations):
    checks = {}
    confidence = safe_float(payload.get("confidence"), 0)

    if "requires_evidence" in expectations:
        checks["requires_evidence"] = (
            bool(payload.get("evidence_ids")) == expectations["requires_evidence"]
        )
    if "requires_human_review" in expectations:
        checks["requires_human_review"] = (
            payload.get("requires_human_review")
            == expectations["requires_human_review"]
        )
    if "minimum_confidence" in expectations:
        checks["minimum_confidence"] = (
            confidence >= safe_float(expectations["minimum_confidence"], 0)
        )
    if "maximum_confidence" in expectations:
        checks["maximum_confidence"] = (
            confidence <= safe_float(expectations["maximum_confidence"], 1)
        )

    failed_checks = [name for name, ok in checks.items() if not ok]
    return {
        "checks": checks,
        "failed_checks": failed_checks,
        "passed": all(checks.values()) if checks else True,
    }

## Execução de cada caso

Nesta fase, cada caso é executado de forma isolada para reduzir contaminação de contexto.

Lógica aplicada por caso:
1. reconstruir tools e crew;
2. executar investigação;
3. converter saída em payload estruturado;
4. aplicar validação determinística (schema, evidência e segurança);
5. aplicar validação de cenário (expectativas por propriedades).

Também tratamos falhas de execução por caso sem interromper toda a bateria,
o que melhora confiabilidade operacional da aula.

In [13]:
def run_case(case):
    events = []
    tools = build_project_tools(TARGET_PROJECT, catalog, events)["tools"]
    crew = build_investigation_crew(
        incident=case["incident"],
        tools=tools,
        llm=llm,
        context_policy=(
            "Use tools seletivamente. Cite apenas evidências consultadas. "
            "Quando não houver base suficiente, reduza a confiança e escale."
        ),
        reviewer=True,
        verbose=False,
    )

    try:
        result = crew.kickoff()
        payload = parse_crew_json(result)

        deterministic = evaluate_analysis(payload, valid_evidence_ids)
        scenario = evaluate_case_expectations(payload, case["expectations"])
        return {
            "case_id": case["case_id"],
            "payload": payload,
            "deterministic": deterministic,
            "scenario": scenario,
            "tool_events": events,
            "passed": deterministic.get("accepted", False) and scenario["passed"],
            "token_usage": getattr(result, "token_usage", None),
            "error": None,
        }
    except Exception as exc:
        deterministic_fallback = {
            "accepted": False,
            "schema": {"valid": False},
            "evidence": {"valid": False},
            "safety": {"valid": False},
            "errors": [str(exc)],
        }
        scenario_fallback = {"checks": {}, "failed_checks": ["runtime_error"], "passed": False}
        return {
            "case_id": case["case_id"],
            "payload": {},
            "deterministic": deterministic_fallback,
            "scenario": scenario_fallback,
            "tool_events": events,
            "passed": False,
            "token_usage": None,
            "error": str(exc),
        }

results = [run_case(case) for case in cases]

## Relatório agregado

O relatório consolida os resultados em colunas comparáveis por caso.

Além de flags de aprovação, o objetivo é mostrar o motivo das falhas para facilitar
diagnóstico e revisão em sala.

Com isso, a discussão deixa de ser apenas aprovado ou reprovado e passa a focar
qual dimensão falhou: contrato, evidência, segurança ou expectativa de cenário.

In [14]:
import pandas as pd

report = pd.DataFrame([
    {
        "case_id": r["case_id"],
        "schema_valid": r["deterministic"].get("schema", {}).get("valid", False),
        "evidence_valid": r["deterministic"].get("evidence", {}).get("valid", False),
        "safety_valid": r["deterministic"].get("safety", {}).get("valid", False),
        "scenario_passed": r["scenario"].get("passed", False),
        "scenario_failed_checks": ", ".join(r["scenario"].get("failed_checks", [])),
        "tool_calls": len(r["tool_events"]),
        "error": r.get("error"),
        "passed": r["passed"],
    }
    for r in results
])
report

,case_id,schema_valid,evidence_valid,safety_valid,scenario_passed,scenario_failed_checks,tool_calls,error,passed
0,case_timeout,True,False,True,False,minimum_confidence,5,None,False
1,case_insufficient_information,True,True,True,True,,4,None,True


## LLM-as-a-judge

Um avaliador com LLM pode complementar critérios semânticos, mas não deve substituir fatos
objetivos: arquivo existe, campo está presente, confiança está na faixa e limite foi atingido.

## Exercício

Adicione um caso que exija evidência em código ou configuração e revisão humana quando houver
conflito entre fontes.